# FinRise Risk Management Project - Part 2: Portfolio Construction for Surplus Liquidity

**Course:** Financial Risk Analytics  
**Team:** Umer Raza, Syed Yahya Tariq  

## Overview
Following the Risk Assessment Report, FinRise Digital Finance Ltd. requires efficient management of surplus liquidity. This notebook constructs and analyzes an investment portfolio consisting of 5 PSX-listed stocks, Treasury Bills, and Bank Deposits, while keeping the Investment Policy in mind (Annexure - A)

**Portfolio Assets:**
- **Equities (55%)**: FFC, Lucky, Nestle, SAzgar, Mebl
- **Treasury Bills (30%)**: 12-month at 15% p.a.
- **Bank Deposits (15%)**: Commercial (12% p.a.) and Islamic (11.5% p.a.), with minimum 5% in Islamic.

**Objectives:**
- Construct portfolio respecting sector and allocation limits
- Calculate expected returns, risk, and Sharpe ratio
- Optimize for maximum Sharpe ratio
- Generate summary report for CRO and CFO

## A Portfolio Overview
**Total Portfolio Value:** Rs 2,000,000,000

**Initial Diversification (before optimization):**
- Stocks (55%) - PKR 110,000,000
- Treasury Bills (30%) - PKR 60,000,000  
- Bank Deposits (15%) - PKR 30,000,000


In [2]:
portfolio_Value = 200000000
stocks_allocation = 0.55
treasury_allocation = 0.30
bank_allocation = 0.15
stocks_value = portfolio_Value * stocks_allocation
treasury_value = portfolio_Value * treasury_allocation
bank_value = portfolio_Value * bank_allocation
print(f"Stocks Value: ${stocks_value:,.2f}")
print(f"Treasury Value: ${treasury_value:,.2f}")
print(f"Bank Value: ${bank_value:,.2f}")


Stocks Value: $110,000,000.00
Treasury Value: $60,000,000.00
Bank Value: $30,000,000.00


In [3]:
import pandas as pd

df = pd.read_excel("../data/stock_prices.xlsx")
print (df.head())



        Date     FFC  SAZEW   OGDC    MEBL    LUCK
0 2023-01-02  100.56  48.82  80.22  100.70  439.25
1 2023-01-03  100.19  48.55  79.94   99.76  431.40
2 2023-01-04  101.38  48.22  78.55   98.86  432.61
3 2023-01-05  101.94  50.81  79.77   97.98  430.74
4 2023-01-06  102.30  50.71  82.31   96.43  433.11


In [4]:
print("Shape:", df.shape)
print("\nColumns:", df.columns)

print("\nData Types:")
print(df.dtypes)

print("\nMissing Values:")
print(df.isnull().sum())

Shape: (807, 6)

Columns: Index(['Date', 'FFC', 'SAZEW', 'OGDC', 'MEBL', 'LUCK'], dtype='object')

Data Types:
Date     datetime64[ns]
FFC             float64
SAZEW           float64
OGDC            float64
MEBL            float64
LUCK            float64
dtype: object

Missing Values:
Date     0
FFC      0
SAZEW    0
OGDC     0
MEBL     0
LUCK     5
dtype: int64


In [5]:
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
df = df.sort_values(by='Date')

# Forward fill (use last available price)
df = df.fillna(method='ffill')

# Backward fill (for initial missing values)
df = df.fillna(method='bfill')

C:\Users\syedy\AppData\Local\Temp\ipykernel_28644\3990490617.py:5: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method='ffill')
C:\Users\syedy\AppData\Local\Temp\ipykernel_28644\3990490617.py:8: FutureWarning: DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.
  df = df.fillna(method='bfill')


In [6]:
print("Missing after cleaning:")
print(df.isnull().sum())

Missing after cleaning:
Date     0
FFC      0
SAZEW    0
OGDC     0
MEBL     0
LUCK     0
dtype: int64


In [7]:
df = df.reset_index(drop=True)

print(df.info())
print(df.describe())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 807 entries, 0 to 806
Data columns (total 6 columns):
 #   Column  Non-Null Count  Dtype         
---  ------  --------------  -----         
 0   Date    807 non-null    datetime64[ns]
 1   FFC     807 non-null    float64       
 2   SAZEW   807 non-null    float64       
 3   OGDC    807 non-null    float64       
 4   MEBL    807 non-null    float64       
 5   LUCK    807 non-null    float64       
dtypes: datetime64[ns](1), float64(5)
memory usage: 38.0 KB
None
                                Date         FFC        SAZEW        OGDC  \
count                            807  807.000000   807.000000  807.000000   
mean   2024-08-20 02:24:32.118959360  267.387993   848.674226  169.610409   
min              2023-01-02 00:00:00   90.970000    44.710000   73.690000   
25%              2023-10-26 12:00:00  103.790000   154.060000  101.955000   
50%              2024-08-26 00:00:00  180.540000   988.840000  138.340000   
75%              

In [8]:
df.to_excel("cleaned_stock_prices.xlsx", index=False)